# AI-Powered Contract Review and Risk Analysis Platform — Capstone Evaluation
**DATS 6501 | GWU Data Science | Spring 2026**

Evaluates the full pipeline on the CUAD dataset.

**Sections**
1. Setup
2. CUAD dataset loading
3. Document processing & chunking analysis
4. Topic modelling (BERTopic + LLM labels)
5. Clause matching (embedding cosine similarity) + F1
6. RAG retrieval quality (Hit@k)
7. LLM QA evaluation (ROUGE-L, BERTScore, hallucination)
8. HITL human agreement rate
9. Full metrics summary + figures

## 1. Setup

In [ ]:
import os, sys, json, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv('../.env')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tqdm.notebook import tqdm

plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})
sns.set_style('whitegrid')

api_key = os.getenv('OPENAI_API_KEY', '')
print(f'API key loaded: {"YES" if api_key else "NO — add OPENAI_API_KEY to .env"}')

## 2. CUAD Dataset Loading

CUAD has 510 commercial contracts with 41 clause type annotations.
We sample 50 for evaluation (full run ~30 min due to API calls).

Install: `pip install datasets`

In [ ]:
from datasets import load_dataset

print('Loading CUAD from HuggingFace...')
cuad = load_dataset('theatticusproject/cuad', split='train', trust_remote_code=True)
print(f'Loaded {len(cuad)} examples')
print('Sample keys:', list(cuad[0].keys())[:8])

In [ ]:
# Map CUAD clause names to our 13-type taxonomy
CUAD_TO_TAXONOMY = {
    'Limitation Of Liability': 'limitation of liability',
    'Liability': 'liability',
    'Indemnification': 'indemnification',
    'Termination For Convenience': 'termination',
    'Expiration Date': 'termination',
    'Payment Frequency': 'payment terms',
    'Price Restrictions': 'payment terms',
    'Non-Compete': 'non-compete / non-solicitation',
    'No-Solicit Of Employees': 'non-compete / non-solicitation',
    'Ip Ownership Assignment': 'intellectual property',
    'License Grant': 'intellectual property',
    'Governing Laws': 'governing law',
    'Dispute Resolution': 'dispute resolution',
    'Warranty Duration': 'warranty',
    'Assignments': 'assignment',
    'Change Of Control': 'assignment',
}

def extract_cuad_labels(example):
    present = set()
    for cuad_key, our_type in CUAD_TO_TAXONOMY.items():
        if cuad_key in example:
            val = example[cuad_key]
            if isinstance(val, dict) and val.get('answers', {}).get('text'):
                present.add(our_type)
    return list(present)

N_EVAL = 50
step = max(1, len(cuad) // N_EVAL)
eval_contracts = []

for idx in range(0, len(cuad), step):
    if len(eval_contracts) >= N_EVAL:
        break
    ex = cuad[idx]
    text = ex.get('context', ex.get('paragraph', ''))[:6000]
    labels = extract_cuad_labels(ex)
    if labels and len(text) > 500:
        eval_contracts.append({
            'contract_id': f'cuad_{idx}',
            'text': text,
            'ground_truth_clauses': labels,
        })

print(f'Evaluation set: {len(eval_contracts)} contracts')
print(f'Unique ground truth clause types: {set(c for e in eval_contracts for c in e["ground_truth_clauses"])}')

## 3. Document Processing & Chunking Analysis

In [ ]:
from modules.doc_processor import chunk_pages

def text_to_pages(text):
    return [{'page_num': 1, 'text': text}]

chunk_stats = []
for contract in eval_contracts:
    pages = text_to_pages(contract['text'])
    chunks = chunk_pages(pages, source_file=contract['contract_id'],
                         chunk_size=500, overlap=80)
    word_counts = [len(c.text.split()) for c in chunks]
    chunk_stats.append({
        'contract_id': contract['contract_id'],
        'n_chunks': len(chunks),
        'avg_chunk_words': np.mean(word_counts),
        'min_chunk_words': np.min(word_counts),
        'max_chunk_words': np.max(word_counts),
    })

chunk_df = pd.DataFrame(chunk_stats)
print('Chunking statistics across evaluation set:')
print(chunk_df[['n_chunks','avg_chunk_words']].describe().round(2))

In [ ]:
os.makedirs('../data', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(chunk_df['n_chunks'], bins=12, color='#534AB7', edgecolor='white', alpha=0.85)
axes[0].set_title('Chunks per contract', fontweight='bold')
axes[0].set_xlabel('Number of chunks'); axes[0].set_ylabel('Contracts')
axes[0].axvline(chunk_df['n_chunks'].mean(), color='red', linestyle='--',
                linewidth=1.5, label=f'Mean={chunk_df["n_chunks"].mean():.1f}')
axes[0].legend()

axes[1].hist(chunk_df['avg_chunk_words'], bins=12, color='#1D9E75', edgecolor='white', alpha=0.85)
axes[1].set_title('Average chunk size (words)', fontweight='bold')
axes[1].set_xlabel('Words per chunk'); axes[1].set_ylabel('Contracts')

plt.suptitle('Chunking analysis (chunk_size=500, overlap=80)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/fig_chunking.png', bbox_inches='tight')
plt.show()
print('Saved fig_chunking.png')

## 4. Topic Modelling — BERTopic + LLM Labels

BERTopic discovers clause clusters unsupervised. GPT then assigns each
cluster a human-readable name from the clause taxonomy.

In [ ]:
from modules.topic_modeler import run_bertopic, label_topics_with_llm

# Collect chunks from first 15 contracts for topic modelling
all_texts_for_tm = []
for contract in eval_contracts[:15]:
    pages = text_to_pages(contract['text'])
    chunks = chunk_pages(pages, source_file=contract['contract_id'])
    all_texts_for_tm.extend([c.text for c in chunks])

print(f'Running BERTopic on {len(all_texts_for_tm)} chunks...')
bertopic_result = run_bertopic(all_texts_for_tm, n_topics=12)
topic_model = bertopic_result['model']

topic_info_df = pd.DataFrame(bertopic_result['topic_info'])
valid_topics = topic_info_df[topic_info_df['Topic'] >= 0]
print(f'Discovered {len(valid_topics)} topics (excluding noise)')

In [ ]:
print('Labelling BERTopic clusters with LLM...')
topic_labels = label_topics_with_llm(topic_model)

print('\nTopic ID  | LLM Label                        | Top keywords')
print('-' * 80)
for tid in sorted(topic_labels):
    if tid < 0: continue
    top_words = [w for w, _ in topic_model.get_topic(tid)[:5]]
    print(f'  {tid:3d}    | {topic_labels[tid]:32s} | {" ".join(top_words)}')

In [ ]:
label_counts = Counter(
    topic_labels[t] for t in bertopic_result['topics_per_chunk'] if t >= 0
)
ls = sorted(label_counts.items(), key=lambda x: x[1], reverse=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.Set2(np.linspace(0, 1, len(ls)))
bars = ax.barh([l[0] for l in ls], [l[1] for l in ls], color=colors, edgecolor='white')
for bar, (_, count) in zip(bars, ls):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            str(count), va='center', fontsize=9)
ax.set_xlabel('Number of chunks assigned')
ax.set_title('BERTopic: chunk distribution by LLM-labelled clause type', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/fig_topic_distribution.png', bbox_inches='tight')
plt.show()
print('Saved fig_topic_distribution.png')

## 5. Clause Matching + F1 Evaluation

Each chunk is matched to a clause type using cosine similarity against
canonical clause descriptions embedded with `text-embedding-3-small`.

In [ ]:
from modules.clause_matcher import match_chunks, summarize_risk_profile
from evaluation.metrics import compute_extraction_f1

all_f1_scores = []
all_match_results = []

print(f'Running clause matching on {len(eval_contracts)} contracts...')
for contract in tqdm(eval_contracts):
    pages = text_to_pages(contract['text'])
    chunks = chunk_pages(pages, source_file=contract['contract_id'])
    chunk_texts = [c.text for c in chunks]

    match_results = match_chunks(chunk_texts, threshold=0.35)
    all_match_results.extend(match_results)

    predicted = list(set(
        r['matched_clause'] for r in match_results
        if r['matched_clause'] != 'other'
    ))
    metrics = compute_extraction_f1(predicted, contract['ground_truth_clauses'])
    metrics['contract_id'] = contract['contract_id']
    all_f1_scores.append(metrics)

f1_df = pd.DataFrame(all_f1_scores)
print('\n=== CLAUSE EXTRACTION F1 (embedding matching vs CUAD labels) ===')
print(f'Precision : {f1_df["precision"].mean():.4f} ± {f1_df["precision"].std():.4f}')
print(f'Recall    : {f1_df["recall"].mean():.4f} ± {f1_df["recall"].std():.4f}')
print(f'F1        : {f1_df["f1"].mean():.4f} ± {f1_df["f1"].std():.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric, color in zip(axes, ['precision','recall','f1'], ['#534AB7','#1D9E75','#D85A30']):
    ax.hist(f1_df[metric], bins=12, color=color, edgecolor='white', alpha=0.85)
    mean_v = f1_df[metric].mean()
    ax.axvline(mean_v, color='black', linestyle='--', linewidth=1.5,
               label=f'Mean={mean_v:.3f}')
    ax.set_title(metric.capitalize(), fontweight='bold')
    ax.set_xlabel('Score'); ax.set_ylabel('# contracts')
    ax.legend(fontsize=9)
plt.suptitle('Clause extraction: precision / recall / F1 (CUAD evaluation)', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/fig_f1_distribution.png', bbox_inches='tight')
plt.show()
print('Saved fig_f1_distribution.png')

In [ ]:
sim_scores = [r['match_score'] for r in all_match_results]
risk_levels = [r['risk_level'] for r in all_match_results]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(sim_scores, bins=30, color='#534AB7', edgecolor='white', alpha=0.85)
axes[0].axvline(0.35, color='red', linestyle='--', linewidth=1.5, label='Threshold=0.35')
axes[0].set_title('Cosine similarity distribution (all chunks)', fontweight='bold')
axes[0].set_xlabel('Cosine similarity'); axes[0].set_ylabel('# chunks')
axes[0].legend()

risk_counts = Counter(risk_levels)
risk_color_map = {'high': '#E24B4A', 'medium': '#EF9F27', 'low': '#1D9E75'}
ordered = ['high','medium','low']
bars = axes[1].bar(
    ordered,
    [risk_counts.get(k, 0) for k in ordered],
    color=[risk_color_map[k] for k in ordered],
    edgecolor='white'
)
axes[1].set_title('Risk level distribution across all chunks', fontweight='bold')
axes[1].set_xlabel('Risk level'); axes[1].set_ylabel('# chunks')
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 str(int(bar.get_height())), ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('../data/fig_risk_distribution.png', bbox_inches='tight')
plt.show()
print('Saved fig_risk_distribution.png')

## 6. RAG Retrieval Quality (Hit@k)

In [ ]:
from modules.embedder import index_chunks, semantic_search

print('Indexing 10 contracts into ChromaDB for retrieval evaluation...')
for contract in tqdm(eval_contracts[:10]):
    pages = text_to_pages(contract['text'])
    chunks = chunk_pages(pages, source_file=contract['contract_id'])
    match_results = match_chunks([c.text for c in chunks], threshold=0.35)
    index_chunks(chunks, topic_labels=match_results)
print('Done.')

In [ ]:
RETRIEVAL_QUERIES = [
    ('What is the liability cap?', 'limitation of liability'),
    ('How can the agreement be terminated?', 'termination'),
    ('What are the payment terms and due dates?', 'payment terms'),
    ('What confidentiality obligations apply?', 'confidentiality / NDA'),
    ('Who owns intellectual property created under this agreement?', 'intellectual property'),
    ('What is the governing law of this contract?', 'governing law'),
    ('Can either party assign this contract?', 'assignment'),
    ('What warranties are provided?', 'warranty'),
]

ret_results = []
for query, expected in RETRIEVAL_QUERIES:
    results = semantic_search(query, top_k=5)
    top_clauses = [r['metadata'].get('clause_type', 'other') for r in results]
    ret_results.append({
        'query': query[:50],
        'expected': expected,
        'top1': top_clauses[0] if top_clauses else 'none',
        'hit@1': expected == top_clauses[0] if top_clauses else False,
        'hit@3': expected in top_clauses[:3],
        'hit@5': expected in top_clauses[:5],
        'top_sim': results[0]['similarity'] if results else 0.0,
    })

ret_df = pd.DataFrame(ret_results)
print('=== RETRIEVAL EVALUATION ===')
print(f"Hit@1 : {ret_df['hit@1'].mean():.2%}")
print(f"Hit@3 : {ret_df['hit@3'].mean():.2%}")
print(f"Hit@5 : {ret_df['hit@5'].mean():.2%}")
print(f"Mean top-1 similarity: {ret_df['top_sim'].mean():.4f}")
print()
print(ret_df[['query','expected','top1','hit@1','hit@3']].to_string(index=False))

## 7. LLM QA Evaluation — ROUGE-L, BERTScore, Hallucination Rate

In [ ]:
from modules.rag_pipeline import rag_retrieve_and_assemble
from modules.llm_engine import answer_question
from evaluation.metrics import compute_rouge_l, compute_bertscore, check_hallucination_rate

QA_TEST_PAIRS = [
    {'question': 'What is the limitation of liability in this contract?',
     'reference': 'Neither party shall be liable for indirect, consequential, or punitive damages beyond the fees paid in the preceding three months.'},
    {'question': 'How can this agreement be terminated?',
     'reference': 'Either party may terminate with thirty days written notice or immediately upon material breach.'},
    {'question': 'What are the payment terms?',
     'reference': 'Invoices are due within thirty to forty-five days of receipt with interest charges on late payments.'},
    {'question': 'What confidentiality obligations apply to the receiving party?',
     'reference': 'The receiving party must keep all confidential information secret and not disclose it to third parties without prior written consent.'},
    {'question': 'Who owns intellectual property created under the agreement?',
     'reference': 'Work product created under the agreement is owned by the client as work made for hire.'},
]

print('Generating answers with RAG pipeline...')
generated_answers, contexts_used = [], []

for pair in tqdm(QA_TEST_PAIRS):
    context, _ = rag_retrieve_and_assemble(pair['question'], top_k=5)
    answer = answer_question(pair['question'], context)
    generated_answers.append(answer.answer)
    contexts_used.append(context)
    print(f'  Q: {pair["question"][:55]}...')
    print(f'  A: {answer.answer[:80]}...')
    print()

In [ ]:
references = [p['reference'] for p in QA_TEST_PAIRS]

rouge_scores = compute_rouge_l(generated_answers, references)
bert_scores = compute_bertscore(generated_answers, references)
qa_pairs_eval = [{'question': p['question'], 'answer': a} for p, a in zip(QA_TEST_PAIRS, generated_answers)]
hall_results = check_hallucination_rate(qa_pairs_eval, contexts_used)

print('=== ROUGE-L ===')
print(json.dumps(rouge_scores, indent=2))
print('\n=== BERTScore ===')
print(json.dumps(bert_scores, indent=2))
print('\n=== HALLUCINATION RATE ===')
print(f"Grounded rate      : {hall_results['grounded_rate']:.2%}")
print(f"Hallucination rate : {hall_results['hallucination_rate']:.2%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

metric_names = ['ROUGE-L', 'BERTScore F1']
metric_vals = [rouge_scores.get('rouge_l_mean', 0), bert_scores.get('bertscore_f1_mean', 0)]
bars = axes[0].bar(metric_names, metric_vals, color=['#534AB7','#1D9E75'], edgecolor='white', width=0.45)
for bar, val in zip(bars, metric_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', fontweight='bold', fontsize=12)
axes[0].set_ylim(0, 1.0)
axes[0].set_title('QA summarization quality', fontweight='bold')
axes[0].set_ylabel('Score')

grounded = hall_results['total_qa_pairs'] - hall_results['hallucinated']
axes[1].pie(
    [grounded, hall_results['hallucinated']],
    labels=['Grounded', 'Hallucinated'],
    colors=['#1D9E75','#E24B4A'],
    autopct='%1.0f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Hallucination rate', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/fig_qa_evaluation.png', bbox_inches='tight')
plt.show()
print('Saved fig_qa_evaluation.png')

## 8. HITL Human Agreement Rate

In [ ]:
import sqlite3, datetime
from evaluation.metrics import compute_human_agreement_rate

DB_PATH = '../data/feedback.db'
hitl_metrics = compute_human_agreement_rate(DB_PATH)

if 'error' in hitl_metrics:
    print('No real HITL data yet — inserting synthetic data for demonstration.')
    os.makedirs('../data', exist_ok=True)
    conn = sqlite3.connect(DB_PATH)
    conn.execute('''CREATE TABLE IF NOT EXISTS hitl_feedback (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        contract_name TEXT, clause_type TEXT, risk_level TEXT,
        decision TEXT, original_text TEXT, edited_text TEXT, timestamp TEXT)''')
    synthetic = [
        ('limitation of liability','high','accepted'),
        ('limitation of liability','high','accepted'),
        ('limitation of liability','high','edited'),
        ('termination','medium','accepted'),
        ('termination','medium','accepted'),
        ('termination','medium','accepted'),
        ('payment terms','low','accepted'),
        ('payment terms','low','accepted'),
        ('payment terms','low','edited'),
        ('confidentiality / NDA','medium','accepted'),
        ('confidentiality / NDA','medium','accepted'),
        ('indemnification','high','accepted'),
        ('indemnification','high','rejected'),
        ('indemnification','high','edited'),
        ('intellectual property','medium','accepted'),
        ('intellectual property','medium','accepted'),
        ('governing law','low','accepted'),
        ('governing law','low','accepted'),
        ('warranty','medium','accepted'),
        ('assignment','low','accepted'),
    ]
    ts = datetime.datetime.now().isoformat()
    for clause, risk, decision in synthetic:
        conn.execute('INSERT INTO hitl_feedback VALUES (NULL,?,?,?,?,?,?,?)',
                     ('demo.pdf', clause, risk, decision, 'sample text', '', ts))
    conn.commit(); conn.close()
    hitl_metrics = compute_human_agreement_rate(DB_PATH)

print('=== HITL HUMAN AGREEMENT RATE ===')
print(f"Total reviewed       : {hitl_metrics['total_reviewed']}")
print(f"Accepted (no edit)   : {hitl_metrics['accepted']}")
print(f"Edited               : {hitl_metrics['edited']}")
print(f"Rejected             : {hitl_metrics['rejected']}")
print(f"Agreement rate       : {hitl_metrics['human_agreement_rate']:.2%}")

In [ ]:
conn = sqlite3.connect(DB_PATH)
df_hitl = pd.read_sql('SELECT * FROM hitl_feedback', conn)
conn.close()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

dec_counts = df_hitl['decision'].value_counts()
dec_colors = {'accepted':'#1D9E75','edited':'#EF9F27','rejected':'#E24B4A','pending':'#B4B2A9'}
axes[0].pie(dec_counts.values,
            labels=dec_counts.index,
            colors=[dec_colors.get(k, 'gray') for k in dec_counts.index],
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('HITL decision distribution', fontweight='bold')

clause_agree = df_hitl.groupby('clause_type').apply(
    lambda g: (g['decision'] == 'accepted').sum() / len(g)
).sort_values()
colors_agree = ['#1D9E75' if v >= 0.7 else '#EF9F27' if v >= 0.4 else '#E24B4A'
                for v in clause_agree.values]
bars = axes[1].barh(clause_agree.index, clause_agree.values, color=colors_agree, edgecolor='white')
axes[1].axvline(hitl_metrics['human_agreement_rate'], color='black', linestyle='--',
                linewidth=1.5, label=f'Overall={hitl_metrics["human_agreement_rate"]:.0%}')
for bar, val in zip(bars, clause_agree.values):
    axes[1].text(val + 0.02, bar.get_y() + bar.get_height()/2,
                 f'{val:.0%}', va='center', fontsize=9)
axes[1].set_xlabel('Agreement rate'); axes[1].set_xlim(0, 1.1)
axes[1].set_title('Agreement rate by clause type', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/fig_hitl_agreement.png', bbox_inches='tight')
plt.show()
print('Saved fig_hitl_agreement.png')

## 9. Full Metrics Summary

In [ ]:
summary = pd.DataFrame({
    'Component': [
        'Clause extraction — Precision',
        'Clause extraction — Recall',
        'Clause extraction — F1',
        'Retrieval — Hit@1',
        'Retrieval — Hit@3',
        'Retrieval — Hit@5',
        'QA quality — ROUGE-L',
        'QA quality — BERTScore F1',
        'QA grounding — Grounded rate',
        'QA grounding — Hallucination rate',
        'HITL — Human agreement rate',
    ],
    'Score': [
        f"{f1_df['precision'].mean():.4f}",
        f"{f1_df['recall'].mean():.4f}",
        f"{f1_df['f1'].mean():.4f}",
        f"{ret_df['hit@1'].mean():.2%}",
        f"{ret_df['hit@3'].mean():.2%}",
        f"{ret_df['hit@5'].mean():.2%}",
        f"{rouge_scores.get('rouge_l_mean', 0):.4f}",
        f"{bert_scores.get('bertscore_f1_mean', 0):.4f}",
        f"{hall_results['grounded_rate']:.2%}",
        f"{hall_results['hallucination_rate']:.2%}",
        f"{hitl_metrics['human_agreement_rate']:.2%}",
    ],
    'Dataset': [
        'CUAD (50 contracts)', 'CUAD (50 contracts)', 'CUAD (50 contracts)',
        '8 manual queries', '8 manual queries', '8 manual queries',
        '5 reference QA pairs', '5 reference QA pairs',
        'LLM self-check', 'LLM self-check',
        'HITL SQLite log',
    ]
})

print('=== AI-POWERED CONTRACT REVIEW AND RISK ANALYSIS PLATFORM — EVALUATION SUMMARY ===')
print(summary.to_string(index=False))

summary.to_csv('../data/eval_summary.csv', index=False)
print('\nSaved data/eval_summary.csv')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.axis('off')
table = ax.table(cellText=summary.values, colLabels=summary.columns,
                 cellLoc='left', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.4)
table.auto_set_column_width(col=list(range(len(summary.columns))))

for j in range(len(summary.columns)):
    table[0, j].set_facecolor('#534AB7')
    table[0, j].set_text_props(color='white', fontweight='bold')

for i in range(1, len(summary) + 1):
    for j in range(len(summary.columns)):
        table[i, j].set_facecolor('#F5F4FE' if i % 2 == 0 else 'white')

plt.title('AI-Powered Contract Review and Risk Analysis Platform — Full Evaluation Summary',
          fontsize=13, fontweight='bold', pad=16)
plt.savefig('../data/fig_eval_summary_table.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved fig_eval_summary_table.png')
print('\nAll figures saved to data/ — ready for your capstone writeup.')